
# LangChain Basics: Document Loading and Text Splitting

Work done in this notebook:

- Install and import LangChain libraries.
- Load a PDF document using LangChain's Document Loader.
- Extract and inspect the document's text and metadata.
- Split the extracted text into smaller chunks using a Text Splitter.
- Experiment with chunk size and chunk overlap.
- Inspect the resulting chunks.

The goal is to understand how LangChain processes documents before they are used in a RAG application.


###Installing Required Libraries

LangChain's document loading and text splitting components will be used here:
- **langchain-community**: Provides integrations with external tools and document loaders.
- **langchain-text-splitters**: Provides text splitting functionality.
- **pypdf**: Helps extract text from PDF files.

In [1]:
!pip install -q langchain-community langchain-text-splitters pypdf

In [2]:
#importing libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_2935/530829966.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
#loading a sample pdf
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/drylab.pdf")
documents = loader.load()

print("Number of pages:", len(documents))

Number of pages: 3


In [4]:
# Inspect the first page
print(documents[0])

page_content='DrylabNewsfor investors & friends · May 2017
Welcome to our first newsletter of 2017! It's
been a while since the last one, and a lot has
happened. We promise to keep them coming
every two months hereafter, and permit
ourselves to make this one rather long. The
big news is the beginnings of our launch in
the American market, but there are also
interesting updates on sales, development,
mentors and (of course) the investment
round that closed in January.
New capital: The investment round was
successful. We raised 2.13 MNOK to match
the 2.05 MNOK loan from Innovation
Norway. Including the development
agreement with Filmlance International, the
total new capital is 5 MNOK, partly tied to
the successful completion of milestones. All
formalities associated with this process are
now finalized.
New owners: We would especially like to
warmly welcome our new owners to the
Drylab family: Unni Jacobsen, Torstein Jahr,
Suzanne Bolstad, Eivind Bergene, Turid Brun,
Vigdis Trondsen, Lea

Here, documents[0] contains a Document object with two main parts:
1. **page_content**: the actual text extracted from page 1 of your PDF.
2. **metadata**: information about the document:
   - source: /content/drylab.pdf , where the PDF is stored.
   - page: 0 [page index].
   - total_pages: 3

So, LangChain has loaded the PDF as 3 separate Document objects, one for each page. Accessing the content and metadata separately:

In [5]:
#prints the first 500 characters of the extracted text
print("Extracted text:")
print(documents[0].page_content[:500])

print("\nMetadata:")
print(documents[0].metadata)

Extracted text:
DrylabNewsfor investors & friends · May 2017
Welcome to our first newsletter of 2017! It's
been a while since the last one, and a lot has
happened. We promise to keep them coming
every two months hereafter, and permit
ourselves to make this one rather long. The
big news is the beginnings of our launch in
the American market, but there are also
interesting updates on sales, development,
mentors and (of course) the investment
round that closed in January.
New capital: The investment round was
succ

Metadata:
{'producer': 'Prince 16 (www.princexml.com)', 'creator': 'PyPDF', 'creationdate': '', 'source': '/content/drylab.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


### Text Splitters

LangChain tools used to divide large documents into smaller pieces called chunks. The following values are used:


*   **chunk_size=500:** Each chunk can contain up to 500 characters.
*   **chunk_overlap=50:** Consecutive chunks can share 50 characters.




In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split the loaded documents into smaller chunks
chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 15


### Inspecting the Chunks

After splitting, LangChain returns a list of Document objects called chunks.

Each chunk contains:
- **page_content:** The text contained in that chunk.
- **metadata:** Information about its original source and page.

Inspecting the first chunk to understand how the original document has been divided.

In [7]:
# Display the first chunk
print(chunks[0].page_content)

# Display its metadata
print("\nMetadata:")
print(chunks[0].metadata)

DrylabNewsfor investors & friends · May 2017
Welcome to our first newsletter of 2017! It's
been a while since the last one, and a lot has
happened. We promise to keep them coming
every two months hereafter, and permit
ourselves to make this one rather long. The
big news is the beginnings of our launch in
the American market, but there are also
interesting updates on sales, development,
mentors and (of course) the investment
round that closed in January.
New capital: The investment round was

Metadata:
{'producer': 'Prince 16 (www.princexml.com)', 'creator': 'PyPDF', 'creationdate': '', 'source': '/content/drylab.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


Observation: splitting does not discard the original metadata. Each chunk retains information about where it came from.

### Experimenting with Chunk Size

Chunk size determines how much text is included in each chunk.
Comparing the number of chunks generated using different chunk sizes.

In [8]:
# Experiment with different chunk sizes
for size in [200, 500, 1000]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=50
    )

    split_chunks = splitter.split_documents(documents)

    print(f"Chunk size: {size} | Total chunks: {len(split_chunks)}")

Chunk size: 200 | Total chunks: 43
Chunk size: 500 | Total chunks: 15
Chunk size: 1000 | Total chunks: 8


### Experimenting with Chunk Overlap

Chunk overlap determines how much text is repeated between consecutive chunks.

For example, if chunk overlap is 50, approximately 50 characters from the end of one chunk are repeated at the beginning of the next.

- Smaller overlap means less repeated text.
- Larger overlap means more repeated text.

Overlap helps preserve context when a sentence or topic continues across chunk boundaries. Comparing different overlap values while keeping the chunk size constant.

In [9]:
# Experiment with different overlap values
for overlap in [0, 50, 100]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=overlap
    )

    split_chunks = splitter.split_documents(documents)

    print(f"Chunk overlap: {overlap} | Total chunks: {len(split_chunks)}")

Chunk overlap: 0 | Total chunks: 13
Chunk overlap: 50 | Total chunks: 15
Chunk overlap: 100 | Total chunks: 16


### Observations

1. Reducing chunk size increases the number of chunks because the document is divided into smaller pieces.
2. Increasing chunk size decreases the number of chunks because each chunk can hold more text.
3. Increasing chunk overlap generally increases the number of chunks because more text is repeated between consecutive chunks.
4. Chunk size and overlap affect how document content is divided and how much context is preserved.

These parameters need to be chosen carefully in RAG applications because they influence the quality of information available during retrieval.